# LeetCode 1472 — Design Browser History bằng Python

Link đề: https://leetcode.com/problems/design-browser-history/ · Độ khó: **Medium** · Chủ đề: Array, Linked List, Stack, Design, Doubly-Linked List, Data Stream

Skeleton LeetCode:

```python
class BrowserHistory:

    def __init__(self, homepage: str):
        ...

    def visit(self, url: str) -> None:
        ...

    def back(self, steps: int) -> str:
        ...

    def forward(self, steps: int) -> str:
        ...


# Your BrowserHistory object will be instantiated and called as such:
# obj = BrowserHistory(homepage)
# obj.visit(url)
# param_2 = obj.back(steps)
# param_3 = obj.forward(steps)
```

> Đây là bài **thiết kế lớp**. Trong notebook ta tự dựng một **bộ chạy chuỗi lệnh giả lập** kiểu LeetCode (nhận danh sách tên lệnh + danh sách tham số, trả về danh sách kết quả) để kiểm thử — đó là bản giả lập, không phải trình chấm thật.

## 1. Mục tiêu học

1. Mô hình hóa lịch sử trình duyệt một tab: **trang hiện tại**, các trang **lùi** được và các trang **tiến** được; hiểu vì sao `visit` phải **xóa lịch sử tiến**.
2. Cài đặt bằng **hai stack** (`back` / `forward`) — cách trực quan nhất.
3. Cài đặt bằng **danh sách liên kết đôi** — mỗi trang biết trang trước và sau.
4. Tối ưu thành **mảng + con trỏ `cur` + biên phải `last`**: mọi thao tác `O(1)`, không thật sự xóa phần tử.
5. Viết **bộ chạy lệnh giả lập** để test lớp thiết kế giống trình chấm LeetCode, và đối chiếu ngẫu nhiên nhiều cài đặt.

## 2. Đề bài & ví dụ

Trình duyệt có một tab, bắt đầu ở `homepage`. Cài đặt lớp `BrowserHistory`:
- `BrowserHistory(homepage)`: khởi tạo với trang chủ.
- `visit(url)`: từ trang hiện tại, mở `url`. **Xóa toàn bộ lịch sử tiến** (forward).
- `back(steps)`: lùi tối đa `steps` bước (nếu chỉ lùi được `x < steps` bước thì lùi `x`). Trả về URL hiện tại sau khi lùi.
- `forward(steps)`: tiến tối đa `steps` bước, tương tự. Trả về URL hiện tại.

**Ví dụ:**

```
Input:
["BrowserHistory","visit","visit","visit","back","back","forward","visit","forward","back","back"]
[["leetcode.com"],["google.com"],["facebook.com"],["youtube.com"],[1],[1],[1],["linkedin.com"],[2],[2],[7]]
Output:
[null,null,null,null,"facebook.com","google.com","facebook.com",null,"linkedin.com","google.com","leetcode.com"]
```

Giải thích: sau ba lần `visit`, lịch sử là `leetcode → google → facebook → youtube` (đang ở youtube). `back(1)` → facebook, `back(1)` → google, `forward(1)` → facebook. `visit("linkedin.com")` xóa `youtube` khỏi lịch sử tiến. `forward(2)` không tiến được → linkedin. `back(2)` → google. `back(7)` chỉ lùi được 1 bước → leetcode.

**Ràng buộc:**
- `1 <= homepage.length, url.length <= 20`, chỉ gồm `'.'` và chữ thường
- `1 <= steps <= 100`
- Tối đa `5000` lời gọi `visit`, `back`, `forward`.

In [1]:
import random
import time
from typing import List, Optional

OPS = ["BrowserHistory", "visit", "visit", "visit", "back", "back", "forward", "visit", "forward", "back", "back"]
ARGS = [["leetcode.com"], ["google.com"], ["facebook.com"], ["youtube.com"], [1], [1], [1],
        ["linkedin.com"], [2], [2], [7]]
EXPECTED = [None, None, None, None, "facebook.com", "google.com", "facebook.com", None,
            "linkedin.com", "google.com", "leetcode.com"]

def run_ops(cls, ops, args):
    """Bộ chạy lệnh GIẢ LẬP kiểu LeetCode: lệnh đầu là constructor, các lệnh sau gọi method."""
    obj = cls(*args[0])
    out = [None]
    for name, a in zip(ops[1:], args[1:]):
        out.append(getattr(obj, name)(*a))
    return out

print(list(zip(OPS, ARGS, EXPECTED)))

[('BrowserHistory', ['leetcode.com'], None), ('visit', ['google.com'], None), ('visit', ['facebook.com'], None), ('visit', ['youtube.com'], None), ('back', [1], 'facebook.com'), ('back', [1], 'google.com'), ('forward', [1], 'facebook.com'), ('visit', ['linkedin.com'], None), ('forward', [2], 'linkedin.com'), ('back', [2], 'google.com'), ('back', [7], 'leetcode.com')]


## 3. Kiến thức nền

### 3.1 Lớp (class), `__init__` và thuộc tính

Lớp gom **dữ liệu** (thuộc tính `self.x`) và **hành vi** (method). `__init__` chạy khi tạo đối tượng. Mỗi đối tượng có bộ thuộc tính riêng.

In [2]:
class Counter:
    def __init__(self, start):
        self.value = start
    def inc(self, d=1):
        self.value += d
        return self.value

c1, c2 = Counter(0), Counter(100)
print(c1.inc(), c1.inc(5), c2.inc())

1 6 101


### 3.2 List làm stack

`append(x)` đẩy lên đỉnh, `pop()` lấy ra từ đỉnh, `stack[-1]` xem đỉnh — đều `O(1)`. `clear()` xóa hết (tốn `O(số phần tử)`).

In [3]:
st = []
for x in "abc":
    st.append(x)
print("đỉnh:", st[-1], "| pop:", st.pop(), "| còn:", st)
st.clear()
print("sau clear:", st)

đỉnh: c | pop: c | còn: ['a', 'b']
sau clear: []


### 3.3 `getattr` — gọi method theo tên (chuỗi)

Bộ chạy lệnh giả lập nhận tên method dạng chuỗi (`"visit"`, `"back"`…). `getattr(obj, "visit")` trả về method tương ứng để gọi.

In [4]:
c = Counter(10)
m = getattr(c, "inc")
print(m(3), getattr(c, "value"))

13 13


### 3.4 Kẹp giá trị (clamp) bằng `min` / `max`

"Lùi tối đa `steps` nhưng không quá đầu lịch sử" ⇔ `cur = max(0, cur − steps)`. "Tiến tối đa nhưng không quá cuối" ⇔ `cur = min(last, cur + steps)`.

In [5]:
cur, last = 2, 5
print("back(7):", max(0, cur - 7), "| back(1):", max(0, cur - 1))
print("forward(10):", min(last, cur + 10), "| forward(2):", min(last, cur + 2))

back(7): 0 | back(1): 1
forward(10): 5 | forward(2): 4


### 3.5 Chỉ số âm trong Python — cái bẫy im lặng

`a[-1]` là phần tử **cuối**, không phải lỗi. Nếu quên kẹp `cur` về `0`, `hist[cur]` với `cur` âm vẫn trả về một trang (sai) mà **không báo lỗi**.

In [6]:
hist = ["leetcode.com", "google.com", "facebook.com"]
cur = 2 - 4      # lùi 4 bước mà không kẹp
print("cur =", cur, "-> hist[cur] =", hist[cur], "(sai! phải là leetcode.com)")

cur = -2 -> hist[cur] = google.com (sai! phải là leetcode.com)


### 3.6 Nút danh sách liên kết đôi

Mỗi nút giữ `val`, `prev`, `next`. Cắt lịch sử tiến chỉ cần gán `cur.next = new_node` — các nút phía sau không còn được tham chiếu và sẽ bị bộ thu gom rác giải phóng.

In [7]:
class Node:
    __slots__ = ("val", "prev", "next")
    def __init__(self, val, prev=None):
        self.val, self.prev, self.next = val, prev, None

a = Node("a"); b = Node("b", a); a.next = b
print(a.val, "->", a.next.val, "| b.prev =", b.prev.val)

a -> b | b.prev = a


## 4. Ý tưởng nền tảng / trực giác

Hình dung lịch sử là một **dãy trang** và một **con trỏ** chỉ trang hiện tại. Mô phỏng tay ví dụ (dấu `*` là trang hiện tại):

| lệnh | dãy trang | trả về |
|---|---|---|
| khởi tạo | `leetcode*` | – |
| visit google | `leetcode, google*` | – |
| visit facebook | `leetcode, google, facebook*` | – |
| visit youtube | `leetcode, google, facebook, youtube*` | – |
| back 1 | `leetcode, google, facebook*, youtube` | facebook |
| back 1 | `leetcode, google*, facebook, youtube` | google |
| forward 1 | `leetcode, google, facebook*, youtube` | facebook |
| visit linkedin | `leetcode, google, facebook, linkedin*` (youtube bị xóa) | – |
| forward 2 | không có trang tiến | linkedin |
| back 2 | `leetcode, google*, facebook, linkedin` | google |
| back 7 | `leetcode*, …` (chỉ lùi được 1) | leetcode |

Quan sát: `back`/`forward` chỉ **dời con trỏ**; `visit` **cắt** phần sau con trỏ rồi nối trang mới.

In [8]:
def simulate_by_hand(ops, args):
    pages, cur = [args[0][0]], 0
    out = [None]
    for name, (x,) in zip(ops[1:], args[1:]):
        if name == "visit":
            pages = pages[:cur + 1] + [x]; cur += 1; out.append(None)
        elif name == "back":
            cur = max(0, cur - x); out.append(pages[cur])
        else:
            cur = min(len(pages) - 1, cur + x); out.append(pages[cur])
        view = ", ".join(p.split(".")[0] + ("*" if i == cur else "") for i, p in enumerate(pages))
        print(f"{name:8s} {x!s:13s} -> [{view}]")
    return out

print(simulate_by_hand(OPS, ARGS) == EXPECTED)

visit    google.com    -> [leetcode, google*]
visit    facebook.com  -> [leetcode, google, facebook*]
visit    youtube.com   -> [leetcode, google, facebook, youtube*]
back     1             -> [leetcode, google, facebook*, youtube]
back     1             -> [leetcode, google*, facebook, youtube]
forward  1             -> [leetcode, google, facebook*, youtube]
visit    linkedin.com  -> [leetcode, google, facebook, linkedin*]
forward  2             -> [leetcode, google, facebook, linkedin*]
back     2             -> [leetcode, google*, facebook, linkedin]
back     7             -> [leetcode*, google, facebook, linkedin]
True


## 5. Cách 1 — Brute Force / baseline: list + con trỏ, cắt bằng slicing

**Ý tưởng:** giữ `self.pages` (list) và `self.cur`. `visit` cắt bỏ phần sau `cur` (`del pages[cur+1:]`) rồi `append`. `back/forward` kẹp con trỏ.

Đúng, ngắn, dễ hiểu — dùng làm chuẩn đối chiếu.

In [9]:
class BrowserHistoryBaseline:
    def __init__(self, homepage: str):
        self.pages = [homepage]
        self.cur = 0

    def visit(self, url: str) -> None:
        del self.pages[self.cur + 1:]          # xóa lịch sử tiến: O(số trang bị xóa)
        self.pages.append(url)
        self.cur += 1

    def back(self, steps: int) -> str:
        self.cur = max(0, self.cur - steps)
        return self.pages[self.cur]

    def forward(self, steps: int) -> str:
        self.cur = min(len(self.pages) - 1, self.cur + steps)
        return self.pages[self.cur]

got = run_ops(BrowserHistoryBaseline, OPS, ARGS)
print(got)
print("khớp kỳ vọng:", got == EXPECTED)

[None, None, None, None, 'facebook.com', 'google.com', 'facebook.com', None, 'linkedin.com', 'google.com', 'leetcode.com']
khớp kỳ vọng: True


**Độ phức tạp:** `back`/`forward` `O(1)`; `visit` `O(t)` với `t` là số trang bị xóa (tổng phân bổ vẫn `O(1)` vì mỗi trang chỉ bị xóa một lần sau khi được thêm). Bộ nhớ `O(số trang)`.

**Nút thắt:** về lý thuyết đã tốt. Nhưng nếu ta **sao chép** list thay vì `del` (ví dụ `self.pages = self.pages[:cur+1] + [url]`) thì mỗi `visit` tốn `O(n)` — dễ thành `O(n²)`. Cell dưới so sánh hai kiểu.

In [10]:
class BrowserHistoryCopy(BrowserHistoryBaseline):
    def visit(self, url: str) -> None:
        self.pages = self.pages[:self.cur + 1] + [url]   # tạo list mới mỗi lần: O(n)
        self.cur += 1

for cls in (BrowserHistoryBaseline, BrowserHistoryCopy):
    t = time.perf_counter()
    b = cls("home")
    for i in range(20000):
        b.visit(f"p{i}")
    print(f"{cls.__name__:24s}: 20000 visit liên tiếp {time.perf_counter() - t:.3f}s")

BrowserHistoryBaseline  : 20000 visit liên tiếp 0.005s


BrowserHistoryCopy      : 20000 visit liên tiếp 1.089s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (hai stack):** những trang "lùi được" xếp thành một chồng — trang gần nhất ở trên; những trang "tiến được" cũng thế. `back` = chuyển trang từ đỉnh chồng lùi sang chồng tiến; `visit` = đẩy vào chồng lùi và **vứt** chồng tiến.

**Insight 2 (không cần xóa thật):** với mảng + con trỏ, ta không cần xóa các trang sau `cur` khi `visit`; chỉ cần nhớ **biên phải hợp lệ** `last`. Trang mới **ghi đè** vào ô `cur + 1` và đặt `last = cur + 1`. Các ô sau `last` là rác, không bao giờ được đọc.

**Quy tắc:**
- `visit`: `cur += 1`; **nếu** `cur == len(hist)` **thì** `append`, **ngược lại** ghi đè `hist[cur]`; `last = cur`.
- `back(s)`: `cur = max(0, cur − s)`.
- `forward(s)`: `cur = min(last, cur + s)`.

**Invariant:** `hist[0..last]` là lịch sử hợp lệ theo thứ tự, `0 ≤ cur ≤ last`, và `hist[cur]` là trang hiện tại.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai stack

**Từng đoạn:**
1. `self.back_st = [homepage]` (đỉnh là trang hiện tại), `self.fwd_st = []`.
2. `visit(url)`: `back_st.append(url)`, `fwd_st.clear()`.
3. `back(steps)`: trong khi `steps > 0` và `len(back_st) > 1` (phải giữ lại trang hiện tại!): chuyển đỉnh `back_st` sang `fwd_st`. Trả về `back_st[-1]`.
4. `forward(steps)`: trong khi `steps > 0` và `fwd_st` khác rỗng: chuyển đỉnh `fwd_st` sang `back_st`.

In [11]:
class BrowserHistoryTwoStacks:
    def __init__(self, homepage: str):
        self.back_st = [homepage]     # đỉnh = trang hiện tại
        self.fwd_st = []

    def visit(self, url: str) -> None:
        self.back_st.append(url)
        self.fwd_st.clear()

    def back(self, steps: int) -> str:
        while steps and len(self.back_st) > 1:
            self.fwd_st.append(self.back_st.pop())
            steps -= 1
        return self.back_st[-1]

    def forward(self, steps: int) -> str:
        while steps and self.fwd_st:
            self.back_st.append(self.fwd_st.pop())
            steps -= 1
        return self.back_st[-1]

print(run_ops(BrowserHistoryTwoStacks, OPS, ARGS) == EXPECTED)

True


Code nộp LeetCode (Cách 2):

In [12]:
class BrowserHistory:

    def __init__(self, homepage: str):
        self.back_st = [homepage]
        self.fwd_st = []

    def visit(self, url: str) -> None:
        self.back_st.append(url)
        self.fwd_st.clear()

    def back(self, steps: int) -> str:
        while steps and len(self.back_st) > 1:
            self.fwd_st.append(self.back_st.pop())
            steps -= 1
        return self.back_st[-1]

    def forward(self, steps: int) -> str:
        while steps and self.fwd_st:
            self.back_st.append(self.fwd_st.pop())
            steps -= 1
        return self.back_st[-1]

print(run_ops(BrowserHistory, OPS, ARGS))

[None, None, None, None, 'facebook.com', 'google.com', 'facebook.com', None, 'linkedin.com', 'google.com', 'leetcode.com']


**Độ phức tạp:** `visit` `O(1)` phân bổ (`clear` tốn theo số trang bị xóa), `back/forward` `O(min(steps, số trang))` ≤ `O(100)`. Bộ nhớ `O(số trang)`.

### 7.2 Cách 3 — Danh sách liên kết đôi

**Từng đoạn:**
1. `self.cur = Node(homepage)`.
2. `visit(url)`: tạo `node = Node(url, prev=self.cur)`, gán `self.cur.next = node` (tự động "cắt" lịch sử tiến cũ), `self.cur = node`.
3. `back(steps)`: đi theo `prev` tối đa `steps` lần, dừng nếu `prev is None`.
4. `forward(steps)`: tương tự theo `next`.

In [13]:
class BrowserHistoryLinked:
    def __init__(self, homepage: str):
        self.cur = Node(homepage)

    def visit(self, url: str) -> None:
        node = Node(url, self.cur)
        self.cur.next = node          # bỏ toàn bộ chuỗi tiến cũ
        self.cur = node

    def back(self, steps: int) -> str:
        while steps and self.cur.prev:
            self.cur = self.cur.prev
            steps -= 1
        return self.cur.val

    def forward(self, steps: int) -> str:
        while steps and self.cur.next:
            self.cur = self.cur.next
            steps -= 1
        return self.cur.val

print(run_ops(BrowserHistoryLinked, OPS, ARGS) == EXPECTED)

True


Code nộp LeetCode (Cách 3, kèm lớp nút):

In [14]:
class _Node:
    __slots__ = ("val", "prev", "next")
    def __init__(self, val, prev=None):
        self.val, self.prev, self.next = val, prev, None

class BrowserHistory:

    def __init__(self, homepage: str):
        self.cur = _Node(homepage)

    def visit(self, url: str) -> None:
        node = _Node(url, self.cur)
        self.cur.next = node
        self.cur = node

    def back(self, steps: int) -> str:
        while steps and self.cur.prev:
            self.cur = self.cur.prev
            steps -= 1
        return self.cur.val

    def forward(self, steps: int) -> str:
        while steps and self.cur.next:
            self.cur = self.cur.next
            steps -= 1
        return self.cur.val

print(run_ops(BrowserHistory, OPS, ARGS))

[None, None, None, None, 'facebook.com', 'google.com', 'facebook.com', None, 'linkedin.com', 'google.com', 'leetcode.com']


**Độ phức tạp:** `visit` `O(1)` thật sự (không cần xóa tường minh), `back/forward` `O(steps)`. Bộ nhớ `O(số trang còn tham chiếu)`.

### 7.3 Cách 4 — Mảng + con trỏ `cur` + biên phải `last` (mọi thao tác `O(1)`)

**Từng đoạn:**
1. `self.hist = [homepage]`, `self.cur = 0`, `self.last = 0`.
2. `visit(url)`: `cur += 1`; nếu `cur == len(hist)` thì `append(url)` còn không thì **ghi đè** `hist[cur] = url`; `last = cur`.
3. `back(steps)`: `cur = max(0, cur − steps)`.
4. `forward(steps)`: `cur = min(last, cur + steps)`.

In [15]:
class BrowserHistoryArray:
    def __init__(self, homepage: str):
        self.hist = [homepage]
        self.cur = 0
        self.last = 0

    def visit(self, url: str) -> None:
        self.cur += 1
        if self.cur == len(self.hist):
            self.hist.append(url)
        else:
            self.hist[self.cur] = url      # ghi đè, không xóa phần sau
        self.last = self.cur               # mọi ô > last coi như không tồn tại

    def back(self, steps: int) -> str:
        self.cur = max(0, self.cur - steps)
        return self.hist[self.cur]

    def forward(self, steps: int) -> str:
        self.cur = min(self.last, self.cur + steps)
        return self.hist[self.cur]

print(run_ops(BrowserHistoryArray, OPS, ARGS) == EXPECTED)

True


Code nộp LeetCode (Cách 4):

In [16]:
class BrowserHistory:

    def __init__(self, homepage: str):
        self.hist = [homepage]
        self.cur = 0
        self.last = 0

    def visit(self, url: str) -> None:
        self.cur += 1
        if self.cur == len(self.hist):
            self.hist.append(url)
        else:
            self.hist[self.cur] = url
        self.last = self.cur

    def back(self, steps: int) -> str:
        self.cur = max(0, self.cur - steps)
        return self.hist[self.cur]

    def forward(self, steps: int) -> str:
        self.cur = min(self.last, self.cur + steps)
        return self.hist[self.cur]

print(run_ops(BrowserHistory, OPS, ARGS))

[None, None, None, None, 'facebook.com', 'google.com', 'facebook.com', None, 'linkedin.com', 'google.com', 'leetcode.com']


**Độ phức tạp:** mọi thao tác `O(1)` (không phụ thuộc `steps`). Bộ nhớ `O(số lần visit tối đa)` — các ô rác vẫn nằm trong mảng nhưng sẽ bị ghi đè dần.

So sánh tốc độ trên chuỗi thao tác ngẫu nhiên lớn:

In [17]:
def random_ops(n_calls, rng, n_urls=50, max_steps=100):
    ops, args = ["BrowserHistory"], [["home.com"]]
    for _ in range(n_calls):
        r = rng.random()
        if r < 0.4:
            ops.append("visit"); args.append([f"u{rng.randrange(n_urls)}.com"])
        elif r < 0.7:
            ops.append("back"); args.append([rng.randint(1, max_steps)])
        else:
            ops.append("forward"); args.append([rng.randint(1, max_steps)])
    return ops, args

IMPLS = [BrowserHistoryBaseline, BrowserHistoryTwoStacks, BrowserHistoryLinked, BrowserHistoryArray]
rng = random.Random(1472)
ops_big, args_big = random_ops(200000, rng)
for cls in IMPLS:
    t = time.perf_counter()
    run_ops(cls, ops_big, args_big)
    print(f"{cls.__name__:24s}: {time.perf_counter() - t:.3f}s (200000 lệnh)")

BrowserHistoryBaseline  : 0.048s (200000 lệnh)
BrowserHistoryTwoStacks : 0.052s (200000 lệnh)
BrowserHistoryLinked    : 0.066s (200000 lệnh)
BrowserHistoryArray     : 0.046s (200000 lệnh)


## 8. TRACE từng bước

Trace Cách 4 trên ví dụ của đề. Bảng trạng thái (`hist` chỉ in tới cuối mảng, ô sau `last` là rác):

| lệnh | cur | last | hist | trả về |
|---|---|---|---|---|
| init leetcode | 0 | 0 | [leetcode] | – |
| visit google | 1 | 1 | [leetcode, google] | – |
| visit facebook | 2 | 2 | [.., facebook] | – |
| visit youtube | 3 | 3 | [.., youtube] | – |
| back 1 | 2 | 3 | | facebook |
| back 1 | 1 | 3 | | google |
| forward 1 | 2 | 3 | | facebook |
| visit linkedin | 3 | 3 | [leetcode, google, facebook, linkedin] (ghi đè youtube) | – |
| forward 2 | 3 | 3 | | linkedin |
| back 2 | 1 | 3 | | google |
| back 7 | 0 | 3 | | leetcode |

In [18]:
class TracedArray(BrowserHistoryArray):
    def _show(self, op, ret):
        cells = [(p.split(".")[0] + ("*" if i == self.cur else "")) if i <= self.last else f"({p.split('.')[0]})"
                 for i, p in enumerate(self.hist)]
        print(f"{op:16s} cur={self.cur} last={self.last} hist=[{', '.join(cells)}] -> {ret}")

    def visit(self, url):
        super().visit(url); self._show(f"visit {url.split('.')[0]}", None)

    def back(self, steps):
        r = super().back(steps); self._show(f"back {steps}", r); return r

    def forward(self, steps):
        r = super().forward(steps); self._show(f"forward {steps}", r); return r

out = run_ops(TracedArray, OPS, ARGS)
print("khớp:", out == EXPECTED)

visit google     cur=1 last=1 hist=[leetcode, google*] -> None
visit facebook   cur=2 last=2 hist=[leetcode, google, facebook*] -> None
visit youtube    cur=3 last=3 hist=[leetcode, google, facebook, youtube*] -> None
back 1           cur=2 last=3 hist=[leetcode, google, facebook*, youtube] -> facebook.com
back 1           cur=1 last=3 hist=[leetcode, google*, facebook, youtube] -> google.com
forward 1        cur=2 last=3 hist=[leetcode, google, facebook*, youtube] -> facebook.com
visit linkedin   cur=3 last=3 hist=[leetcode, google, facebook, linkedin*] -> None
forward 2        cur=3 last=3 hist=[leetcode, google, facebook, linkedin*] -> linkedin.com
back 2           cur=1 last=3 hist=[leetcode, google*, facebook, linkedin] -> google.com
back 7           cur=0 last=3 hist=[leetcode*, google, facebook, linkedin] -> leetcode.com
khớp: True


Trace Cách 2 (hai stack) để thấy trang di chuyển giữa hai chồng:

In [19]:
def trace_two_stacks(ops, args):
    b = BrowserHistoryTwoStacks(*args[0])
    print(f"init: back_st={b.back_st} fwd_st={b.fwd_st}")
    for name, a in zip(ops[1:], args[1:]):
        r = getattr(b, name)(*a)
        print(f"{name:8s}{a[0]!s:13s} back_st={[p.split('.')[0] for p in b.back_st]} "
              f"fwd_st={[p.split('.')[0] for p in b.fwd_st]} -> {r}")

trace_two_stacks(OPS, ARGS)

init: back_st=['leetcode.com'] fwd_st=[]
visit   google.com    back_st=['leetcode', 'google'] fwd_st=[] -> None
visit   facebook.com  back_st=['leetcode', 'google', 'facebook'] fwd_st=[] -> None
visit   youtube.com   back_st=['leetcode', 'google', 'facebook', 'youtube'] fwd_st=[] -> None
back    1             back_st=['leetcode', 'google', 'facebook'] fwd_st=['youtube'] -> facebook.com
back    1             back_st=['leetcode', 'google'] fwd_st=['youtube', 'facebook'] -> google.com
forward 1             back_st=['leetcode', 'google', 'facebook'] fwd_st=['youtube'] -> facebook.com
visit   linkedin.com  back_st=['leetcode', 'google', 'facebook', 'linkedin'] fwd_st=[] -> None
forward 2             back_st=['leetcode', 'google', 'facebook', 'linkedin'] fwd_st=[] -> linkedin.com
back    2             back_st=['leetcode', 'google'] fwd_st=['linkedin', 'facebook'] -> google.com
back    7             back_st=['leetcode'] fwd_st=['linkedin', 'facebook', 'google'] -> leetcode.com


## 9. Code nộp LeetCode

Bản khuyên dùng (Cách 4 — mảng + con trỏ + biên phải, mọi thao tác `O(1)`), đúng tên lớp và chữ ký:

In [20]:
class BrowserHistory:

    def __init__(self, homepage: str):
        self.hist = [homepage]
        self.cur = 0          # chỉ số trang hiện tại
        self.last = 0         # chỉ số trang cuối cùng còn hợp lệ

    def visit(self, url: str) -> None:
        self.cur += 1
        if self.cur == len(self.hist):
            self.hist.append(url)
        else:
            self.hist[self.cur] = url
        self.last = self.cur

    def back(self, steps: int) -> str:
        self.cur = max(0, self.cur - steps)
        return self.hist[self.cur]

    def forward(self, steps: int) -> str:
        self.cur = min(self.last, self.cur + steps)
        return self.hist[self.cur]

assert run_ops(BrowserHistory, OPS, ARGS) == EXPECTED
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Mô hình chuẩn:** lịch sử là dãy `P[0..L]` và chỉ số `c`. `visit(u)`: `P := P[0..c] + [u]`, `c := c+1`. `back(s)`: `c := max(0, c−s)`. `forward(s)`: `c := min(L, c+s)`. Kết quả là `P[c]`. (Đây đúng là Cách 1.)

- **Cách 4:** Bất biến `hist[0..last] = P`, `cur = c`. Sau `visit`: ô `cur+1` nhận `u`, `last = cur+1` ⇒ `hist[0..last] = P[0..c] + [u]` (các ô sau `last` không thuộc `P`). `back/forward` giống hệt mô hình vì `last = L`.
- **Cách 2:** Bất biến: `back_st` (từ đáy lên đỉnh) `= P[0..c]`, `fwd_st` (từ đỉnh xuống đáy) `= P[c+1..L]`. `visit` đẩy `u` lên `back_st` và xóa `fwd_st` ⇒ đúng `P[0..c]+[u]`. Mỗi bước `back` chuyển `P[c]` sang đỉnh `fwd_st` ⇒ `c` giảm 1; điều kiện `len(back_st) > 1` ⇔ `c > 0`. Tương tự `forward`.
- **Cách 3:** Chuỗi nút từ đầu tới nút cuối (theo `next`) là `P`; `cur` là nút `P[c]`. `visit` gán `cur.next = node mới` ⇒ chuỗi thành `P[0..c] + [u]`.

## 11. Phân tích Time / Space

Gọi `V` là số lần `visit`, `s ≤ 100` là `steps`.

| Cách | `visit` | `back` / `forward` | Space |
|---|---|---|---|
| 1. List + `del` | `O(1)` phân bổ (`O(t)` khi xóa `t` trang) | `O(1)` | `O(V)` |
| 2. Hai stack | `O(1)` phân bổ (`clear`) | `O(min(s, số trang))` | `O(V)` |
| 3. Liên kết đôi | `O(1)` | `O(min(s, số trang))` | `O(V)` (nút bị cắt được GC thu hồi) |
| 4. Mảng + `cur` + `last` | `O(1)` | `O(1)` | `O(V)` |

Với tối đa 5000 lời gọi, mọi cách đều đủ nhanh; Cách 4 là tối ưu về độ phức tạp từng thao tác.

## 12. Bộ test

Chạy ví dụ đề qua bộ chạy lệnh giả lập cho mọi cài đặt; edge case (chỉ có homepage, back/forward khi không có trang, `visit` cùng URL nhiều lần, `steps` lớn); đối chiếu ngẫu nhiên với Cách 1 (seed cố định, tham số đúng ràng buộc).

In [21]:
ALL_IMPLS = IMPLS + [BrowserHistory]
for cls in ALL_IMPLS:
    assert run_ops(cls, OPS, ARGS) == EXPECTED, cls.__name__

edge_cases = [
    (["BrowserHistory", "back", "forward"], [["a.com"], [5], [5]], [None, "a.com", "a.com"]),
    (["BrowserHistory", "visit", "visit", "back", "visit", "forward", "back"],
     [["a"], ["b"], ["c"], [1], ["d"], [1], [100]], [None, None, None, "b", None, "d", "a"]),
    (["BrowserHistory", "visit", "visit", "back", "back", "forward", "forward"],
     [["x"], ["x"], ["x"], [1], [1], [1], [1]], [None, None, None, "x", "x", "x", "x"]),
    (["BrowserHistory", "visit", "visit", "visit", "back", "visit", "forward", "back"],
     [["h"], ["a"], ["b"], ["c"], [3], ["z"], [2], [1]], [None, None, None, None, "h", None, "z", "h"]),
]
for ops, args, want in edge_cases:
    for cls in ALL_IMPLS:
        got = run_ops(cls, ops, args)
        assert got == want, (cls.__name__, ops, got, want)
print("Ví dụ + edge case: OK cho", [c.__name__ for c in ALL_IMPLS])

Ví dụ + edge case: OK cho ['BrowserHistoryBaseline', 'BrowserHistoryTwoStacks', 'BrowserHistoryLinked', 'BrowserHistoryArray', 'BrowserHistory']


In [22]:
rng = random.Random(2024)
for _ in range(500):
    ops, args = random_ops(rng.randint(1, 60), rng, n_urls=rng.choice([2, 5, 30]), max_steps=rng.choice([3, 100]))
    want = run_ops(BrowserHistoryBaseline, ops, args)
    for cls in ALL_IMPLS:
        assert run_ops(cls, ops, args) == want, cls.__name__
print("Random 500 chuỗi lệnh: OK")

# chạy 5000 lời gọi (giới hạn đề) cho bản nộp
ops5, args5 = random_ops(5000, random.Random(5))
t = time.perf_counter()
r = run_ops(BrowserHistory, ops5, args5)
print(f"5000 lời gọi: {time.perf_counter() - t:.4f}s, khớp baseline: {r == run_ops(BrowserHistoryBaseline, ops5, args5)}")

Random 500 chuỗi lệnh: OK
5000 lời gọi: 0.0010s, khớp baseline: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Ghi đè nhưng quên cập nhật biên phải `last`

Dùng mảng + ghi đè nhưng `forward` lại kẹp theo `len(hist) − 1` ⇒ "tiến" vào trang rác đã bị xóa khỏi lịch sử.

In [23]:
class WrongNoLast:
    def __init__(self, homepage):
        self.hist, self.cur = [homepage], 0
    def visit(self, url):
        self.cur += 1
        if self.cur == len(self.hist):
            self.hist.append(url)
        else:
            self.hist[self.cur] = url          # ghi đè nhưng không giới hạn phía sau
    def back(self, steps):
        self.cur = max(0, self.cur - steps); return self.hist[self.cur]
    def forward(self, steps):
        self.cur = min(len(self.hist) - 1, self.cur + steps)   # SAI: phải là last
        return self.hist[self.cur]

ops = ["BrowserHistory", "visit", "visit", "visit", "back", "visit", "forward"]
args = [["home"], ["a"], ["b"], ["c"], [3], ["x"], [1]]
print("SAI :", run_ops(WrongNoLast, ops, args))
print("đúng:", run_ops(BrowserHistoryArray, ops, args))

SAI : [None, None, None, None, 'home', None, 'b']
đúng: [None, None, None, None, 'home', None, 'x']


**Sửa:** giữ `self.last = self.cur` trong `visit`, và `forward` kẹp theo `last`.

### Lỗi 2 — `visit` nối vào **cuối list** thay vì ngay sau trang hiện tại

In [24]:
class WrongAppendEnd(BrowserHistoryBaseline):
    def visit(self, url):
        self.pages.append(url)                 # SAI: không xóa lịch sử tiến
        self.cur = len(self.pages) - 1

print("SAI :", run_ops(WrongAppendEnd, OPS, ARGS))
print("đúng:", EXPECTED)

SAI : [None, None, None, None, 'facebook.com', 'google.com', 'facebook.com', None, 'linkedin.com', 'facebook.com', 'leetcode.com']
đúng: [None, None, None, None, 'facebook.com', 'google.com', 'facebook.com', None, 'linkedin.com', 'google.com', 'leetcode.com']


Sau `back` về google rồi `visit linkedin`, bản sai vẫn giữ `facebook, youtube` trong lịch sử ⇒ `back(2)` ra `facebook` thay vì `google`. **Sửa:** `del self.pages[self.cur + 1:]` trước khi `append`.

### Lỗi 3 — Không kẹp `cur` về 0 khi `back`

Chỉ số âm trong Python không báo lỗi mà lấy phần tử từ cuối ⇒ sai im lặng.

In [25]:
class WrongNoClamp(BrowserHistoryBaseline):
    def back(self, steps):
        self.cur -= steps                      # SAI: có thể âm
        return self.pages[self.cur]

ops = ["BrowserHistory", "visit", "visit", "back"]
args = [["leetcode.com"], ["google.com"], ["facebook.com"], [4]]
print("SAI :", run_ops(WrongNoClamp, ops, args))
print("đúng:", run_ops(BrowserHistoryBaseline, ops, args))

SAI : [None, None, None, 'google.com']
đúng: [None, None, None, 'leetcode.com']


**Sửa:** `self.cur = max(0, self.cur - steps)`.

### Lỗi 4 — Hai stack: lùi đến khi chồng lùi **rỗng**

Trang hiện tại nằm ở đỉnh `back_st`; nếu pop cả nó thì không còn trang hiện tại.

In [26]:
class WrongEmptyStack(BrowserHistoryTwoStacks):
    def back(self, steps):
        while steps and self.back_st:          # SAI: phải là len(back_st) > 1
            self.fwd_st.append(self.back_st.pop())
            steps -= 1
        return self.back_st[-1] if self.back_st else None

print("SAI :", run_ops(WrongEmptyStack, OPS, ARGS))
print("đúng:", EXPECTED)

SAI : [None, None, None, None, 'facebook.com', 'google.com', 'facebook.com', None, 'linkedin.com', 'google.com', None]
đúng: [None, None, None, None, 'facebook.com', 'google.com', 'facebook.com', None, 'linkedin.com', 'google.com', 'leetcode.com']


`back(7)` cuối cùng pop luôn `leetcode.com` ⇒ trả `None`. **Sửa:** điều kiện `len(self.back_st) > 1`.

### Lỗi 5 — Hai stack: `visit` quên xóa chồng tiến

In [27]:
class WrongKeepForward(BrowserHistoryTwoStacks):
    def visit(self, url):
        self.back_st.append(url)               # SAI: thiếu self.fwd_st.clear()

print("SAI :", run_ops(WrongKeepForward, OPS, ARGS))
print("đúng:", EXPECTED)

SAI : [None, None, None, None, 'facebook.com', 'google.com', 'facebook.com', None, 'youtube.com', 'facebook.com', 'leetcode.com']
đúng: [None, None, None, None, 'facebook.com', 'google.com', 'facebook.com', None, 'linkedin.com', 'google.com', 'leetcode.com']


`forward(2)` sau khi visit linkedin lẽ ra không đi đâu được, nhưng bản sai "tiến" sang `youtube`. **Sửa:** `self.fwd_st.clear()` trong `visit`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Undo/redo", "back/forward", "lịch sử có nhánh bị cắt khi thao tác mới" ⇒ **hai stack** hoặc **mảng + con trỏ + biên**.
- Khi có thao tác "xóa toàn bộ phía sau", hãy nghĩ tới **biên logic** (`last`) thay vì xóa thật ⇒ `O(1)`.
- Bước đi bị giới hạn ("tối đa `steps`") ⇒ **kẹp** bằng `min/max`, cẩn thận chỉ số âm của Python.
- Bài thiết kế: luôn viết **bộ chạy lệnh** `run_ops(cls, ops, args)` để test nhiều cài đặt cùng một lúc.
- Ràng buộc nhỏ (5000 lời gọi, `steps ≤ 100`) ⇒ cách đơn giản nhất cũng qua; chọn cách ít bug nhất trong phỏng vấn, rồi nêu tối ưu.

## 15. Biến thể của bài

1. **Giới hạn dung lượng lịch sử** (chỉ giữ `cap` trang gần nhất) — bài tập 5.
2. **Undo/Redo trình soạn thảo** (ví dụ 2296 Design a Text Editor): hai stack ký tự hai bên con trỏ.
3. **Lịch sử dạng cây** (không xóa nhánh cũ, có thể quay lại nhánh khác) — mỗi nút giữ danh sách con.
4. **Truy vấn "có thể lùi/tiến bao nhiêu bước"**: với Cách 4 là `cur` và `last − cur` trong `O(1)`.
5. **Nhiều tab**: mỗi tab một đối tượng `BrowserHistory`, quản lý bằng dict `tab_id → history`.

## 16. So sánh các cách

| Cách | Ý tưởng | visit | back/forward | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. List + `del` | Cắt list sau `cur` | `O(1)` phân bổ | `O(1)` | `O(V)` | Chuẩn đối chiếu, code nhanh |
| 2. Hai stack | Chuyển trang giữa hai chồng | `O(1)` phân bổ | `O(steps)` | `O(V)` | Gợi ý của đề, dễ giải thích |
| 3. Liên kết đôi | `prev/next` mỗi trang | `O(1)` | `O(steps)` | `O(V)` | Luyện linked list |
| 4. Mảng + `cur` + `last` | Ghi đè + biên logic | `O(1)` | `O(1)` | `O(V)` | Tối ưu, khuyên dùng |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** Chạy chuỗi `["BrowserHistory","visit","back","back","forward","forward","visit","back"]` với `[["a"],["b"],[1],[1],[1],[1],["c"],[2]]`. Kết quả?

**Bài 2 (tính tay):** Với chuỗi bài 1, viết `cur`, `last`, `hist` của Cách 4 sau mỗi lệnh.

**Bài 3 (điền TODO):** Hoàn thiện lớp `BrowserHistoryTodo` (Cách 4) có thêm method `can_back()` trả về số bước lùi được và `can_forward()` trả về số bước tiến được.

**Bài 4 (tự viết lại):** Không nhìn code, viết lại Cách 3 (liên kết đôi) thành lớp `MyLinkedHistory`, rồi đối chiếu ngẫu nhiên với Cách 1.

**Bài 5 (mini challenge):** Viết `CappedHistory(homepage, cap)`: như `BrowserHistory` nhưng chỉ giữ **tối đa `cap` trang** (gồm cả trang hiện tại); khi vượt, trang **cũ nhất** bị quên. Mọi thao tác `O(1)` phân bổ.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
class BrowserHistoryTodo:
    def __init__(self, homepage: str):
        # TODO: hist, cur, last
        raise NotImplementedError
    def visit(self, url: str) -> None:
        raise NotImplementedError
    def back(self, steps: int) -> str:
        raise NotImplementedError
    def forward(self, steps: int) -> str:
        raise NotImplementedError
    def can_back(self) -> int:
        # TODO: số bước lùi được
        raise NotImplementedError
    def can_forward(self) -> int:
        # TODO: số bước tiến được
        raise NotImplementedError

# Bài 5 — TODO
class CappedHistoryTodo:
    def __init__(self, homepage: str, cap: int):
        # TODO: gợi ý dùng collections.deque
        raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Lịch sử `a, b`. (2) Lùi tới `a` rồi không lùi thêm được; tiến tới `b` rồi đứng yên. (3) `visit c` từ `b` ⇒ `a, b, c`, `back(2)` ⇒ `a`.

**Bài 2.** (1) `visit` tăng `cur`, đặt `last = cur`. (2) `back` không đổi `last`. (3) `hist` cuối là `[a, b, c]`.

**Bài 3.** (1) Copy Cách 4. (2) `can_back` là `cur`. (3) `can_forward` là `last − cur`.

**Bài 4.** (1) Lớp nút có `prev`, `next`. (2) `visit` gán `cur.next = node`. (3) Dùng `random_ops` + `run_ops` để so với `BrowserHistoryBaseline`.

**Bài 5.** (1) Dùng `deque` `pages` và chỉ số `cur` trong deque. (2) `visit`: pop phải cho tới khi `len == cur + 1`, rồi `append`, `cur += 1`; nếu `len > cap` thì `popleft` và `cur −= 1`. (3) Đối chiếu với bản tham chiếu dùng list cắt `pages[-cap:]`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1, Bài 2
ops1 = ["BrowserHistory", "visit", "back", "back", "forward", "forward", "visit", "back"]
args1 = [["a"], ["b"], [1], [1], [1], [1], ["c"], [2]]
res1 = run_ops(BrowserHistoryArray, ops1, args1)
print("Bài 1:", res1)
assert res1 == [None, None, "a", "a", "b", "b", None, "a"]

b = BrowserHistoryArray("a")
states = [(b.cur, b.last, list(b.hist))]
for name, a in zip(ops1[1:], args1[1:]):
    getattr(b, name)(*a)
    states.append((b.cur, b.last, list(b.hist)))
print("Bài 2:", states)
assert states[-1] == (0, 2, ["a", "b", "c"])

Bài 1: [None, None, 'a', 'a', 'b', 'b', None, 'a']
Bài 2: [(0, 0, ['a']), (1, 1, ['a', 'b']), (0, 1, ['a', 'b']), (0, 1, ['a', 'b']), (1, 1, ['a', 'b']), (1, 1, ['a', 'b']), (2, 2, ['a', 'b', 'c']), (0, 2, ['a', 'b', 'c'])]


In [30]:
from collections import deque

class BrowserHistoryAns(BrowserHistoryArray):
    def can_back(self) -> int:
        return self.cur
    def can_forward(self) -> int:
        return self.last - self.cur

class MyLinkedHistory:
    class _N:
        __slots__ = ("v", "p", "n")
        def __init__(self, v, p=None):
            self.v, self.p, self.n = v, p, None
    def __init__(self, homepage):
        self.cur = self._N(homepage)
    def visit(self, url):
        nd = self._N(url, self.cur); self.cur.n = nd; self.cur = nd
    def back(self, steps):
        while steps and self.cur.p:
            self.cur = self.cur.p; steps -= 1
        return self.cur.v
    def forward(self, steps):
        while steps and self.cur.n:
            self.cur = self.cur.n; steps -= 1
        return self.cur.v

h = BrowserHistoryAns("a"); h.visit("b"); h.visit("c"); h.back(1)
assert (h.can_back(), h.can_forward()) == (1, 1)
rng = random.Random(77)
for _ in range(300):
    ops, args = random_ops(rng.randint(1, 50), rng, n_urls=6, max_steps=4)
    assert run_ops(MyLinkedHistory, ops, args) == run_ops(BrowserHistoryBaseline, ops, args)
    hb, ref = BrowserHistoryAns(*args[0]), BrowserHistoryBaseline(*args[0])
    for name, a in zip(ops[1:], args[1:]):
        getattr(hb, name)(*a); getattr(ref, name)(*a)
        assert hb.can_back() == ref.cur and hb.can_forward() == len(ref.pages) - 1 - ref.cur
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [31]:
class CappedHistory:
    def __init__(self, homepage: str, cap: int):
        self.pages = deque([homepage])
        self.cur = 0
        self.cap = cap

    def visit(self, url: str) -> None:
        while len(self.pages) > self.cur + 1:
            self.pages.pop()                   # xóa lịch sử tiến
        self.pages.append(url)
        self.cur += 1
        if len(self.pages) > self.cap:
            self.pages.popleft()               # quên trang cũ nhất
            self.cur -= 1

    def back(self, steps: int) -> str:
        self.cur = max(0, self.cur - steps)
        return self.pages[self.cur]

    def forward(self, steps: int) -> str:
        self.cur = min(len(self.pages) - 1, self.cur + steps)
        return self.pages[self.cur]

class CappedRef:
    def __init__(self, homepage, cap):
        self.pages, self.cur, self.cap = [homepage], 0, cap
    def visit(self, url):
        self.pages = self.pages[:self.cur + 1] + [url]
        self.pages = self.pages[-self.cap:]
        self.cur = len(self.pages) - 1
    def back(self, steps):
        self.cur = max(0, self.cur - steps); return self.pages[self.cur]
    def forward(self, steps):
        self.cur = min(len(self.pages) - 1, self.cur + steps); return self.pages[self.cur]

rng = random.Random(15)
for _ in range(400):
    cap = rng.randint(1, 6)
    ops, args = random_ops(rng.randint(1, 60), rng, n_urls=8, max_steps=5)
    a, b2 = CappedHistory(args[0][0], cap), CappedRef(args[0][0], cap)
    for name, x in zip(ops[1:], args[1:]):
        assert getattr(a, name)(*x) == getattr(b2, name)(*x)
print("Bài 5: OK (400 chuỗi lệnh, cap 1..6)")

Bài 5: OK (400 chuỗi lệnh, cap 1..6)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "lịch sử có con trỏ" (O(1) mọi thao tác):**

```python
class BrowserHistory:
    def __init__(self, homepage):
        self.hist, self.cur, self.last = [homepage], 0, 0
    def visit(self, url):
        self.cur += 1
        if self.cur == len(self.hist): self.hist.append(url)
        else: self.hist[self.cur] = url
        self.last = self.cur                       # cắt lịch sử tiến
    def back(self, steps):
        self.cur = max(0, self.cur - steps); return self.hist[self.cur]
    def forward(self, steps):
        self.cur = min(self.last, self.cur + steps); return self.hist[self.cur]
```

**Checklist:**
- [ ] `visit` **xóa lịch sử tiến** (cập nhật `last` / `clear()` / `del`).
- [ ] `back` kẹp về `0` (không để chỉ số âm).
- [ ] `forward` kẹp theo **biên hợp lệ** (`last`), không phải `len(hist) − 1` khi dùng ghi đè.
- [ ] Hai stack: luôn giữ ít nhất 1 trang trong `back_st`.
- [ ] `visit` trả về `None`; `back/forward` trả về URL.
- [ ] Đã chạy bộ chạy lệnh giả lập với ví dụ đề và chuỗi ngẫu nhiên.